# 🧠 Pseudo-Brain: Google Colab Remote Training & Experimentation Pipeline

Turnkey Google Colab notebook for training, self-correction evaluation, and hidden-rule online adaptation benchmarks on premium GPU compute (NVIDIA T4 / V100 / A100 / L4).

### Highlights:
- **One-Click Turnkey Setup**: Connects Google Drive, verifies repository, installs dependencies, and checks GPU.
- **Google Drive Persistence**: Checkpoints, metrics, logs, and benchmark reports survive Colab session disconnections under `MyDrive/PseudoBrain/`.
- **Unattended Background Execution**: Supports closing your browser while Google AI Ultra Colab continues remote training.
- **Resumable Training**: Interrupted runs automatically continue from `checkpoint_latest.pt` via `--resume`.
- **Automated Evaluation**: Immediately evaluates held-out benchmark episodes and produces aggregate CSV and JSON summaries.


## 1. Google Drive Mounting & Directory Setup
Mount Google Drive so that all experiment checkpoints, datasets, logs, and benchmark results persist permanently.


In [ ]:
from google.colab import drive
from pathlib import Path
import os

# Mount Google Drive
drive.mount('/content/drive')

DRIVE_ROOT = Path('/content/drive/MyDrive/PseudoBrain')
for subdir in ['checkpoints', 'runs', 'datasets', 'results', 'experiments', 'logs']:
    (DRIVE_ROOT / subdir).mkdir(parents=True, exist_ok=True)

print(f'✅ Google Drive initialized at: {DRIVE_ROOT}')


## 2. Repository Setup & Python Environment
Clone or update the Pseudo-Brain repository and register module paths.


In [ ]:
import os
import sys
from pathlib import Path

REPO_DIR = Path('/content/Pseudo-Brain')
if not REPO_DIR.exists():
    print('Cloning Pseudo-Brain repository...')
    !git clone https://github.com/defnotean/Pseudo-Brain.git /content/Pseudo-Brain
else:
    print('Repository exists. Pulling latest commits...')
    %cd /content/Pseudo-Brain
    !git pull

%cd /content/Pseudo-Brain

brain_src = str(REPO_DIR / 'brain' / 'src')
brain_exp = str(REPO_DIR / 'brain' / 'experiments')
brain_dir = str(REPO_DIR / 'brain')
repo_root = str(REPO_DIR)

for p in [brain_src, brain_exp, brain_dir, repo_root]:
    if p not in sys.path:
        sys.path.insert(0, p)

print('✅ Paths configured successfully!')


## 3. Dependency Installation
Install the lightweight dependency footprint (`torch`, `numpy`, `scipy`, `scikit-learn`, `pandas`).


In [ ]:
!pip install -q numpy scipy scikit-learn pandas
print('✅ Dependencies installed!')


## 4. Hardware Telemetry & GPU Detection
Inspect available Colab compute (A100, V100, L4, T4) and verify CUDA is active.


In [ ]:
import torch
from irene_brain.device import format_hardware_summary, resolve_device

print('=' * 60)
print('COLAB COMPUTE HARDWARE REPORT')
print('=' * 60)
print(format_hardware_summary())
print('=' * 60)

# Ensure CUDA is available on Colab
device = resolve_device('auto', require_cuda=True)
print(f'✅ Verified active compute accelerator: {device}')


## 5. Dataset Preparation
Verify or generate benchmark corpora directly into Google Drive for persistence.


In [ ]:
from brain.experiments.run import ensure_dataset

DATA_DIR = DRIVE_ROOT / 'datasets'
print(f'Checking datasets in {DATA_DIR}...')
ensure_dataset('online_adaptation', DATA_DIR)
ensure_dataset('memory_benchmark', DATA_DIR)
print('✅ All benchmark datasets ready!')


## 6. Interactive Experiment Launcher & Batch Configuration
Select your experiment suite, model cohort, random seeds, training steps, and whether to resume an interrupted run.


In [ ]:
#@title 🚀 Experiment Configuration { run: 'auto' }
experiment = 'online_adaptation' #@param ['online_adaptation', 'memory_benchmark', 'self_correction']
models = 'reactive, gru, thoughtlet, plastic_thoughtlet' #@param ['reactive', 'gru', 'thoughtlet', 'plastic_thoughtlet', 'reactive, gru, thoughtlet', 'reactive, gru, thoughtlet, plastic_thoughtlet']
seeds = '42, 142, 242' #@param {type:'string'}
steps = 1500 #@param {type:'integer'}
batch_size = 16 #@param {type:'integer'}
resume = False #@param {type:'boolean'}
persist_to_drive = True #@param {type:'boolean'}

output_dir = DRIVE_ROOT / 'runs' if persist_to_drive else Path('/content/runs')
model_list = [m.strip() for m in models.split(',') if m.strip()]
seed_list = [int(s.strip()) for s in seeds.split(',') if s.strip()]

print(f'Experiment:      {experiment}')
print(f'Models:          {model_list}')
print(f'Seeds:           {seed_list}')
print(f'Steps:           {steps}')
print(f'Resume:          {resume}')
print(f'Storage Target:  {output_dir}')


## 7. Run Training & Automatic Evaluation
Executes the batch unattended. Progress, ETA, and metrics are printed live and saved to disk. When training finishes, held-out evaluation executes immediately.


In [ ]:
from brain.experiments.run import run_batch

run_batch(
    experiment=experiment,
    models=model_list,
    seeds=seed_list,
    steps=steps,
    batch_size=batch_size,
    device_str='auto',
    output_dir=output_dir,
    data_dir=DATA_DIR,
    resume=resume,
    auto_eval=True,
)


## 8. View Results & Aggregate Matrix
Inspect the generated benchmark metrics table.


In [ ]:
import pandas as pd
from IPython.display import display

agg_csv = output_dir / experiment / 'aggregate_results.csv'
if agg_csv.exists():
    df = pd.read_csv(agg_csv)
    print('=== BENCHMARK AGGREGATE RESULTS ===')
    display(df)
else:
    print(f'No CSV found at {agg_csv}')


## 9. Google AI Ultra Unattended Remote Execution Tips

With a Google AI Ultra subscription, Colab provides persistent background execution:

1. **Launch Training**: Start cell 7.
2. **Close Browser**: You can safely close your browser tab or disconnect from your workstation. The Colab container will continue running remotely.
3. **Resume Anytime**: If a VM reaches its timeout or preempts, reopen this notebook, keep `resume = True` in cell 6, and rerun. It will detect `checkpoint_latest.pt` in Google Drive and continue exactly where it left off!
4. **Sync Results**: All checkpoints, logs, and CSV/JSON files are permanently saved in your Google Drive at `MyDrive/PseudoBrain/`.
